In [32]:
import pandas as pd
import numpy as np

from wordfreq import zipf_frequency

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import pearsonr

In [33]:
df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

In [34]:
def featurize(df):
    words = df["word"].fillna("").astype(str).to_numpy()

    return np.column_stack([
        [len(w) for w in words],
        [zipf_frequency(w, "ro") for w in words],
        [len(set(w)) for w in words],
        [w.islower() for w in words],
    ])

In [35]:
def make_model():
    return HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=42,
    )

In [ ]:
X = featurize(df_train)
X_test = featurize(df_test)
y = df_train["answer"].to_numpy()

In [36]:
n_bins = 10

y_bins = np.digitize(
    y,
    np.quantile(y, np.linspace(0, 1, n_bins + 1)[1:-1])
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [ ]:
oof = np.zeros(len(y))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = make_model()
    model.fit(X_tr, np.log1p(y_tr))

    val_preds = np.expm1(model.predict(X_val))
    val_preds = np.clip(val_preds, 0, None)

    oof[val_idx] = val_preds

    test_fold_preds = np.expm1(model.predict(X_test))
    test_fold_preds = np.clip(test_fold_preds, 0, None)

    test_preds += test_fold_preds / skf.n_splits

    print(f"Fold {fold}: {eval_metric(y_val, val_preds):.4f}")

print("OOF:", eval_metric(y, oof))

In [ ]:
submission = pd.read_csv("sample_output.csv")
submission["answer"] = test_preds
submission.to_csv("submission.csv", index=False)